# Drosophila Brain Cocaine Response - Notebook 2: Clustering & Annotation

Resumes from the Notebook 1 checkpoint (`checkpoint_01_preprocessed.h5ad`) - PCA-ready, HVG-subset AnnData with the full gene set retained in `adata.raw`.

## Setup: reload paths and checkpoint from Notebook 1

In [ ]:
from pathlib import Path
import sys, gc, math

import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(r"D:\bmp\sc_project2_drosophila_brain")
DATA_DIR = PROJECT_ROOT / "data"
RESULT_DIR = PROJECT_ROOT / "results"
FIG_DIR = RESULT_DIR / "figures"
TABLES_DIR = RESULT_DIR / "tables"

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=120, facecolor="white", frameon=False)
sc.settings.figdir = str(FIG_DIR)

checkpoint_path = RESULT_DIR / "checkpoint_01_preprocessed.h5ad"
adata = sc.read_h5ad(checkpoint_path)
print(f"Loaded checkpoint: {adata.n_obs} cells x {adata.n_vars} genes")

## Step 9: Neighbors, UMAP, Leiden clustering (target ~36 clusters)
Run as separate cells so a crash pinpoints exactly which step failed.

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, random_state=RANDOM_STATE)
gc.collect()

In [ ]:
sc.tl.umap(adata, random_state=RANDOM_STATE)
gc.collect()

In [ ]:
sc.tl.leiden(adata, resolution=0.8, key_added="leiden_res_0.8", random_state=RANDOM_STATE)

num_clusters = len(adata.obs["leiden_res_0.8"].unique())
print(f"Identified {num_clusters} clusters (Target: ~36; teammate's full-data run: 28).")

sc.pl.umap(
    adata, color=["leiden_res_0.8", "sex", "treatment", "condition"], ncols=2,
    save="_clusters_metadata.png",
)

## Step 10: Cluster annotation with canonical neurotransmitter markers

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden_res_0.8", method="wilcoxon")
cluster_markers = sc.get.rank_genes_groups_df(adata, group=None)
cluster_markers.to_csv(TABLES_DIR / "cluster_markers_wilcoxon.csv", index=False)

canonical_markers = ["repo", "elav", "ey", "Fas2", "VAChT", "Gad1", "VGlut", "ple", "SerT", "Tdc2"]
available_markers = [g for g in canonical_markers if g in adata.var_names]
missing_markers = [g for g in canonical_markers if g not in adata.var_names]
print("Available markers:", available_markers)
print("Missing markers (not in HVG set):", missing_markers)

if available_markers:
    sc.pl.dotplot(
        adata, var_names=available_markers, groupby="leiden_res_0.8",
        standard_scale="var", save="_canonical_markers.png",
    )

### Note on markers not in the HVG set
A missing marker can still be visualized via the full gene set in `adata.raw`:
```python
sc.pl.umap(adata, color=['ple'], use_raw=True)
```

## Checkpoint: save clustered, annotated-ready AnnData for the next notebook

In [ ]:
checkpoint_path = RESULT_DIR / "checkpoint_02_clustered.h5ad"
adata.write_h5ad(checkpoint_path)
print("Saved checkpoint:", checkpoint_path.resolve())
print(f"{adata.obs['leiden_res_0.8'].nunique()} clusters")